# 08F – Cross-Validation & Stability Analysis (Updated)

This notebook uses **american_bankruptcy_cleaned.csv** and recreates the **same configuration** as the production model.

> **Why isn't `production_bankruptcy_model.joblib` loaded?**
>
> Cross-validation repeatedly trains a fresh model on different folds. An already-trained model cannot be cross-validated because it has already seen a fixed training set. Therefore, this notebook recreates the production model with the **same hyperparameters** to measure stability in an industry-standard manner.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

DATA_PATH='american_bankruptcy_cleaned.csv'

df=pd.read_csv(DATA_PATH)

target='status_label' if 'status_label' in df.columns else 'target'
X=df.drop(columns=[target])
y=df[target].map({'alive':0,'failed':1}) if df[target].dtype=='object' else df[target]

model=RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight='balanced',
    n_jobs=-1
)

cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)

scores=cross_validate(
    model,
    X,
    y,
    cv=cv,
    scoring=['accuracy','precision','recall','f1','roc_auc'],
    n_jobs=-1
)


In [ ]:
results=pd.DataFrame({
    'Fold':range(1,6),
    'Accuracy':scores['test_accuracy'],
    'Precision':scores['test_precision'],
    'Recall':scores['test_recall'],
    'F1':scores['test_f1'],
    'ROC_AUC':scores['test_roc_auc']
})

summary=results.describe().T[['mean','std','min','max']]

results.to_csv('cross_validation_scores.csv',index=False)
summary.to_csv('cross_validation_summary.csv')

plt.figure(figsize=(10,6))
for m in ['Accuracy','Precision','Recall','F1','ROC_AUC']:
    plt.plot(results['Fold'],results[m],marker='o',label=m)

plt.xlabel('Fold')
plt.ylabel('Score')
plt.title('Cross-Validation Stability')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig('cross_validation_stability.png',dpi=300)
plt.show()

summary


## Deliverables

- `cross_validation_scores.csv`
- `cross_validation_summary.csv`
- `cross_validation_stability.png`

This notebook validates that the production model configuration is stable across multiple train/validation splits.